# Dira B'Hagral — Lottery Probability Analysis

**Goal:** Choose 3 cities to maximize the probability of winning at least one apartment raffle.

**Rules:**
- You may choose exactly **3 cities**
- Within each chosen city you may enter **all available raffles**
- Target group: **Young Couple (זוג צעיר)**

**Math:**
For a city with raffles $r_1, r_2, \ldots, r_n$ where $p_i = \min\left(1, \frac{\text{apartments}_i}{\text{registered}_i}\right)$:

$$P(\text{win in city}) = 1 - \prod_{i=1}^{n}(1 - p_i)$$

Optimal 3 cities = top 3 by individual $P(\text{win})$ — this is provably optimal for independent events.

**Before running:** Make sure you have:
1. Run `python scraper.py` at least once
2. Filled in `preference_rank` values in `cities_rank.csv` (1–10, higher = more preferred)

## 1. Setup & Load Data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display

# Import our scraper module
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from scraper import compute_city_probabilities, CITIES_CSV, DATA_DIR

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
LATEST_CSV = os.path.join(DATA_DIR, 'latest.csv')

if not os.path.exists(LATEST_CSV):
    raise FileNotFoundError(
        f"No data found at {LATEST_CSV}.\n"
        "Run:  python scraper.py\n"
        "then re-run this notebook."
    )

raw = pd.read_csv(LATEST_CSV, encoding='utf-8-sig')
scraped_at = raw['scraped_at'].iloc[0] if 'scraped_at' in raw.columns else 'unknown'
print(f"Loaded {len(raw)} raffle entries from: {LATEST_CSV}")
print(f"Scraped at: {scraped_at}")

## 2. Raw Data Explorer

In [ ]:
print("Columns:", list(raw.columns))
print(f"\nShape: {raw.shape}")
display(raw.head(10))

In [ ]:
print("Cities in dataset:")
city_counts = raw.groupby(['city_english', 'city_hebrew']).size().reset_index(name='raffles')
display(city_counts.sort_values('raffles', ascending=False))

## 3. Per-Raffle Probability

$p_i = \min\left(1, \dfrac{\text{apartments}}{\text{registered}}\right)$

If `registered = 0`, we set $p_i = 1$ (no competition).

In [ ]:
df = raw.copy()

# Per-raffle probability
df['p_win_raffle'] = df.apply(
    lambda r: min(1.0, r['apartments'] / r['registered'])
              if r['registered'] > 0
              else (1.0 if r['apartments'] > 0 else 0.0),
    axis=1
)

print("Raffle-level statistics:")
display(
    df[['city_english', 'project_name', 'apartments', 'registered', 'p_win_raffle', 'lottery_date']]
    .sort_values(['city_english', 'p_win_raffle'], ascending=[True, False])
    .reset_index(drop=True)
)

## 4. Per-City Probability

$P(\text{win in city}) = 1 - \prod_{i=1}^{n}(1 - p_i)$

In [ ]:
city_df = compute_city_probabilities(df)

print(f"\nCity probabilities ({len(city_df)} cities):")
display_df = city_df.copy()
display_df['p_win_pct'] = display_df['p_win'].map('{:.2%}'.format)
display(display_df[['city_english', 'city_hebrew', 'raffles', 'total_apartments', 'total_registered', 'p_win_pct']]
        .rename(columns={'p_win_pct': 'P(win)'}))

## 5. Load Your City Preferences

In [ ]:
if not os.path.exists(CITIES_CSV):
    print(f"cities_rank.csv not found at {CITIES_CSV}")
    print("Run `python scraper.py` to generate it, then fill in preference_rank values.")
    prefs = None
else:
    prefs = pd.read_csv(CITIES_CSV, encoding='utf-8-sig')
    filled = prefs['preference_rank'].notna().sum()
    total = len(prefs)
    print(f"Loaded {total} cities, {filled} have preference_rank filled in.")
    if filled == 0:
        print("\n>>> Fill in the 'preference_rank' column in cities_rank.csv (1-10, higher = more preferred)")
    display(prefs)

## 6. Rankings Table

In [ ]:
# Merge probabilities with preferences
if prefs is not None:
    merged = city_df.merge(prefs[['city_hebrew', 'preference_rank']], on='city_hebrew', how='left')
else:
    merged = city_df.copy()
    merged['preference_rank'] = np.nan

# If no preference set, default to 1 (neutral weight) so city is still ranked
# using P(win) alone rather than being excluded entirely
merged['preference_rank'] = pd.to_numeric(merged['preference_rank'], errors='coerce').fillna(1)

# Weighted score = P(win) × preference_rank  (rank=1 → effectively just P(win))
merged['weighted_score'] = merged['p_win'] * merged['preference_rank']

# Rank A: pure probability
merged['rank_probability'] = merged['p_win'].rank(ascending=False, method='min').astype(int)

# Rank B: weighted score
merged['rank_weighted'] = merged['weighted_score'].rank(ascending=False, method='min').astype(int)

display_cols = [
    'rank_probability', 'city_english', 'city_hebrew',
    'raffles', 'general_apartments', 'general_registered',
    'p_win', 'preference_rank', 'weighted_score', 'rank_weighted'
]
# Fall back to total_apartments if general columns not available (old CSV)
for col in ['general_apartments', 'general_registered']:
    if col not in city_df.columns:
        merged[col] = merged['total_apartments'] if 'total_apartments' in merged.columns else 0

out = merged[display_cols].sort_values('rank_probability').reset_index(drop=True)
out['p_win'] = out['p_win'].map('{:.2%}'.format)
out['weighted_score'] = out['weighted_score'].round(3)

print("Full rankings table (P(win) uses general pool only — excludes handicapped/reservist reserved units):")
display(out)

## 7. Optimal 3 Cities

**Why greedy is optimal:** For independent events, choosing the 3 highest individual $P(\text{win})$ values minimises $\prod(1-P_i)$, which directly maximises $P(\text{win at least one})$.

In [ ]:
def joint_p(p_values):
    """P(winning at least one city) given a list of per-city win probabilities."""
    p_lose_all = 1.0
    for p in p_values:
        p_lose_all *= (1.0 - float(p))
    return 1.0 - p_lose_all

# merged['p_win'] is always numeric (out is a separate display copy)
# --- Option A: Top 3 by pure probability ---
top3_prob = merged.nsmallest(3, 'rank_probability')
jp_prob = joint_p(top3_prob['p_win'].tolist())

print("=" * 55)
print("OPTION A — Top 3 by Pure Probability")
print("=" * 55)
for _, row in top3_prob.iterrows():
    print(f"  • {row['city_english']} ({row['city_hebrew']})")
    print(f"    P(win) = {row['p_win']:.2%}  |  "
          f"{row['raffles']} raffles, {row['total_apartments']} apts, "
          f"{row['total_registered']} registered")
print(f"\n  Joint P(winning at least one city) = {jp_prob:.2%}")

# --- Option B: Top 3 by weighted score ---
weighted_available = merged[merged['preference_rank'].notna() & (merged['preference_rank'] > 0)]
if len(weighted_available) >= 3:
    top3_weighted = weighted_available.nlargest(3, 'weighted_score').copy()
    jp_weighted = joint_p(top3_weighted['p_win'].tolist())

    print()
    print("=" * 55)
    print("OPTION B — Top 3 by Weighted Score (P × Preference)")
    print("=" * 55)
    for _, row in top3_weighted.iterrows():
        print(f"  • {row['city_english']} ({row['city_hebrew']})")
        print(f"    P(win) = {row['p_win']:.2%}  |  Preference = {row['preference_rank']}  |  "
              f"Score = {row['weighted_score']:.3f}")
    print(f"\n  Joint P(winning at least one city) = {jp_weighted:.2%}")
else:
    print("\nNot enough cities with preference_rank filled in for weighted ranking.")
    print("Fill in cities_rank.csv and re-run.")

## 8. Visualizations

In [ ]:
# ── Bar chart: P(win) per city ─────────────────────────────────────────────
plot_df = city_df.copy().sort_values('p_win', ascending=True)
top3_names = set(top3_prob['city_english'].values)
colors = ['#2ecc71' if c in top3_names else '#3498db' for c in plot_df['city_english']]

fig, ax = plt.subplots(figsize=(14, max(6, len(plot_df) * 0.35)))
bars = ax.barh(plot_df['city_english'], plot_df['p_win'] * 100, color=colors)

for bar, val in zip(bars, plot_df['p_win']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1%}', va='center', fontsize=9)

ax.set_xlabel('P(win at least one raffle) %')
ax.set_title('Probability of Winning at Least One Raffle per City', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())

from matplotlib.patches import Patch
legend = [Patch(color='#2ecc71', label='Top 3 (recommended)'),
          Patch(color='#3498db', label='Other cities')]
ax.legend(handles=legend, loc='lower right')

plt.tight_layout()
plt.savefig('city_probabilities.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: city_probabilities.png")

In [ ]:
# ── Bar chart: Weighted score (only cities with preference) ───────────────
if len(weighted_available) > 0:
    w_plot = (
        merged[merged['preference_rank'].notna() & (merged['preference_rank'] > 0)]
        .sort_values('weighted_score', ascending=True)
        .copy()
    )

    if len(weighted_available) >= 3:
        top3w_names = set(top3_weighted['city_english'].values)
    else:
        top3w_names = set()

    w_colors = ['#e74c3c' if c in top3w_names else '#e67e22' for c in w_plot['city_english']]

    fig, ax = plt.subplots(figsize=(14, max(5, len(w_plot) * 0.4)))
    bars = ax.barh(w_plot['city_english'], w_plot['weighted_score'], color=w_colors)

    for bar, (_, row) in zip(bars, w_plot.iterrows()):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"score={row['weighted_score']:.2f}  (P={row['p_win']:.1%}, pref={row['preference_rank']:.0f})",
                va='center', fontsize=8.5)

    ax.set_xlabel('Weighted Score = P(win) × Preference Rank')
    ax.set_title('Weighted City Score (Probability × Your Preference)', fontsize=13, fontweight='bold')

    from matplotlib.patches import Patch
    legend = [Patch(color='#e74c3c', label='Top 3 weighted'),
              Patch(color='#e67e22', label='Other ranked cities')]
    ax.legend(handles=legend, loc='lower right')

    plt.tight_layout()
    plt.savefig('city_weighted_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: city_weighted_scores.png")
else:
    print("No cities with preference_rank set — skipping weighted chart.")

## 9. Time Series — Probability Over Time

Shows how $P(\text{win})$ per city changes as more people register. Requires multiple daily scrapes.

In [ ]:
from scraper import compute_city_probabilities_over_time
import glob

n_snapshots = len(glob.glob(os.path.join(DATA_DIR, 'scraped_*.csv')))
print(f"Historical snapshots available: {n_snapshots}")

if n_snapshots < 2:
    print("Run scraper.py daily to build up historical data for this chart.")
else:
    hist_df = compute_city_probabilities_over_time(DATA_DIR)
    print(f"Loaded {len(hist_df)} city×snapshot rows across {hist_df['snapshot_time'].nunique()} snapshots")
    
    # Show top 10 cities by latest P(win) in time series
    latest_top = city_df.head(10)['city_english'].tolist()
    ts_plot = hist_df[hist_df['city_english'].isin(latest_top)].copy()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    for city, grp in ts_plot.groupby('city_english'):
        grp = grp.sort_values('snapshot_time')
        ax.plot(grp['snapshot_time'], grp['p_win'] * 100, marker='o', label=city)
    
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xlabel('Date')
    ax.set_ylabel('P(win) %')
    ax.set_title('P(Win) Over Time — Top 10 Cities', fontsize=13, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig('city_probabilities_over_time.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: city_probabilities_over_time.png")